# Introduction to Profiling — GNU gprof
## Concepts and practice · NCI ARE · 12 cores on one node

We use **prebuilt LULESH executables** from the private `lulesh-gprof` module for `vp91`.
These three exercises establish a baseline, collect `gmon` data and read flat profiles and call graphs. You run the application and `gprof` directly in shell cells.


## Exercise roadmap

| Stop | Time | Task | Evidence to retain |
|---|---|---|---|
| [Setup](#setup) | 5–10 min | Check the ARE allocation and private module | Executable paths and results directory |
| [1. Establish a baseline](#exercise-1) | 10 min | Warm up; repeat the uninstrumented executable | Three times and numerical diagnostics |
| [2. Collect and read a flat profile](#exercise-2) | 10 min | Run the `-pg` executable; inspect raw data | `gmon.PID`, flat profile and expensive functions |
| [3. Follow the call graph](#exercise-3) | 10 min | Read the table and display a diagram | One caller/function/child explanation |


## What does gprof measure?

Gprof combines two mechanisms:

- **Sampling:** periodic observations of the program counter estimate time spent in code. 
- **Instrumentation:** compiling with `-pg` adds entry hooks that count observed caller–callee arcs. These hooks do not time every call from entry to exit. Optimisation can inline functions or change call boundaries.

**Question of the theme:** a function executes a million times but does almost no work per call. Another runs ten times and does expensive calculations. Which could dominate?


### Two matching executables

| Executable | Use | Configuration |
|---|---|---|
| `lulesh2.0-gprof-baseline` | Unprofiled timing | `-O3 -g -DNDEBUG -fno-omit-frame-pointer` |
| `lulesh2.0-gprof` | Collect gprof data | Same options, plus **`-pg` at compilation and linking** |

Both builds use the same source, GCC/Open MPI versions, MPI enabled and OpenMP disabled. Note that Debug symbols (`-g`) and gprof instrumentation (`-pg`) have different purposes.




<a id="setup"></a>
## Setup

`%%bash` runs a cell as shell commands. Each Bash cell has its own shell, so shell variables do not persist between cells. `tee` displays output and saves it to a file. `set -eo pipefail` stops a Bash cell if a command or pipeline fails.


In [ ]:
import os
# the jupyter notebook is launched from your $HOME, change the working directory provided a username directory is created under /scratch/vp91
os.chdir(os.path.expandvars("/scratch/vp91/$USER/intro-to-profiling"))

In [ ]:
%%bash
command -v lulesh2.0-gprof-baseline lulesh2.0-gprof mpirun gprof


In [ ]:
%%bash
echo "Loaded modules: $LOADEDMODULES"
echo "ARE CPU allocation: $PBS_NCPUS"
mpirun --version
gprof --version


In [ ]:
%pip install gprof2dot

### Choose a useful workload

LULESH is a hydrodynamics code. `-s N` gives the local mesh side length; `-i K` caps iterations. All runs below use one rank with the same core binding.

Start at `-s 30 -i 200` and aim for several seconds of calculation. If necessary, increase `-s` or `-i` in **every LULESH command below**, then repeat the baseline. Check actual iterations and numerical diagnostics in the output.


<a id="exercise-1"></a>
## Exercise 1 — Establish a baseline (8 min)

Start with the baseline executable. Inspect the log for the number of ranks, actual iterations and successful completion. 

In [ ]:
%%bash
set -eo pipefail
mpirun -np 1 --map-by slot:PE=1 --bind-to core lulesh2.0-gprof-baseline -s 30 -i 200 | tee baseline-warmup.log


Use LULESH's printed **`Elapsed time`** consistently: it measures the application's calculation interval, not the whole launcher-to-exit duration. Run the three repetitions below sequentially and compare their numerical output.


In [ ]:
%%bash
set -eo pipefail
for run in 1 2 3; do
    mpirun -np 1 --map-by slot:PE=1 --bind-to core lulesh2.0-gprof-baseline -s 30 -i 200 | tee baseline-$run.log
done
grep "Elapsed time" baseline-[123].log


**Checkpoint:** record the three times, timing interval, input and numerical diagnostics. For three times, the median is the middle value after sorting; record the minimum and maximum as the range.


## From executable to report

`gprof` is not a launcher, it only reads data **after** the application exits. An equivalent one-rank collection in a fresh writable directory is:

```bash
export OMP_NUM_THREADS=1 OMP_DYNAMIC=FALSE
export GMON_OUT_PREFIX="$PWD/gmon"
mpirun -np 1 --map-by slot:PE=1 --bind-to core \
  lulesh2.0-gprof -s 30 -i 200
# Use the actual PID-suffixed file written by this run:
gprof -b --demangle -p /path/to/lulesh2.0-gprof /path/to/gmon.PID
```

`GMON_OUT_PREFIX` sets the output prefix; the profiling runtime appends the process ID on normal exit. The cells below use `$(command -v lulesh2.0-gprof)` to locate the same executable used for collection.


<a id="exercise-2"></a>
## Exercise 2 — Collect and read a flat profile 

**Predict:** will the function with the most calls also have the greatest self time?

Run the instrumented executable using exactly the same mesh, iteration cap, rank count and placement as the baseline. Check numerical output before interpreting costs.

In [ ]:
%%bash
set -eo pipefail
export OMP_NUM_THREADS=1 OMP_DYNAMIC=FALSE
export GMON_OUT_PREFIX="$PWD/gmon"
mpirun -np 1 --map-by slot:PE=1 --bind-to core lulesh2.0-gprof -s 30 -i 200 | tee profile.log


### Locate the profile file

LULESH's completion summary does **not** print the profile filename. After the instrumented run finishes, run the next cell to list the actual `gmon.<PID>` files and their full paths. This only lists files; it does not rerun LULESH.


In [ ]:
%%bash
echo "Profile files in $PWD (newest first):"
ls -lht "$PWD"/gmon.[0-9]*


Copy the filename or full path from the **file-listing cell immediately above** into the next cell, replacing `gmon.PID`. For example, if the listing contains `gmon.4080836`, use `%env GMON_FILE=gmon.4080836`. `%env` makes this choice available to all later Bash cells as `$GMON_FILE`.

If the listing says **`No such file or directory`**, no matching profile was found in the printed directory. Check that you ran the collection cell with `lulesh2.0-gprof` (the `-pg` build), that `GMON_OUT_PREFIX` was exported in that same Bash cell, and that the run exited successfully. 

If you repeat collection, select the file from the new successful run; the listing shows the newest files first. Pass one filename to analyse one run. Passing several files to `gprof` would sum their profiles.


In [ ]:
%env GMON_FILE=gmon.PID


In [ ]:
%%bash
gprof -i "$(command -v lulesh2.0-gprof)" "$GMON_FILE"


In [ ]:
%%bash
set -eo pipefail
gprof -b --demangle -p "$(command -v lulesh2.0-gprof)" "$GMON_FILE" | tee flat-profile.txt


### Read the units and denominator

| Field | Interpretation |
|---|---|
| `% time` | Share of the report's attributed sampled time spent in the function itself |
| `self seconds` | Sampled time attributed to that function's own code |
| `cumulative seconds` | Running sum down the flat table; not descendants of this function |
| `calls` | Recorded instrumented calls where available|
| `self .../call` | Self time divided by the recorded calls |
| `total .../call` | Self plus estimated descendant time per call |


**Illustrative arithmetic:** 0.8 self seconds over 400 calls gives 2 ms of average self time per call. If the report attributes 2 seconds in total, this function has 40% self time. 

`-i` describes the raw profile records; `-b` removes explanatory prose; `-p` selects the flat profile; `--demangle` requests readable C++ names. `-C` means execution counts in GNU gprof


**Checkpoint:** record the sampling period, three costly functions, their self time and calls.


<a id="exercise-3"></a>
## Exercise 3 — Follow the call graph 

**Predict:** can a function have almost no self time but a large call-graph percentage?

In each entry, the primary function row starts with `[index]`. Its callers appear above it and its children below. Primary-row `% time` includes **self plus estimated descendants**. Costs overlap across parent/child entries, so these percentages should not be added.

Gprof distributes a callee's cost among callers using call counts. It assumes comparable average cost per call from each caller. If one caller triggers expensive work and another mostly takes an early return, that attribution may mislead. 



In [ ]:
%%bash
set -eo pipefail
gprof -b --demangle -q "$(command -v lulesh2.0-gprof)" "$GMON_FILE" | tee call-graph.txt


### Display the call graph as a diagram

`-q` selects Gprof's **text call graph**. The commands below convert the saved `call-graph.txt` into a diagram without running LULESH again. The table remains available in the notebook and in that text file.

Run the `%pip install gprof2dot` setup cell first. The converter below uses the notebook's Python interpreter (`sys.executable`), so it does not require a `gprof2dot` command on `PATH`. Graphviz's `dot` executable must still be on `PATH`. The `&&` runs Graphviz only if conversion succeeds.

Boxes are functions and arrows are recorded calls. Colours highlight expensive regions. The `-s` option removes C++ argument lists from labels. Set **`SHOW_ALL_FUNCTIONS = True`** below to include low-cost functions and edges (`-n 0 -e 0`), then rerun the conversion and display cells. Leave it `False` for the filtered view. A diagram cannot recover callers reported as `<spontaneous>`.

References: [gprof2dot](https://github.com/jrfonseca/gprof2dot), [Graphviz SVG output](https://graphviz.org/docs/outputs/svg/).


In [ ]:
import sys

SHOW_ALL_FUNCTIONS = False  # Set True to include low-cost functions and edges.
thresholds = "-n 0 -e 0" if SHOW_ALL_FUNCTIONS else ""

!"{sys.executable}" -m gprof2dot -f prof -s {thresholds} call-graph.txt -o call-graph.dot && dot -Tsvg call-graph.dot -o call-graph.svg


In [ ]:
from IPython.display import SVG, display

display(SVG(filename="call-graph.svg"))


### Find a function in the source

Replace the function name below with one from your report. `LULESH_SOURCE` is set by the private module. `grep -n` prints matching filenames and line numbers; this is source lookup, not a measurement of line-level execution time.


In [ ]:
%%bash
grep -n "CalcHourglassControlForElems" "$LULESH_SOURCE"/*.cc


**Checkpoint:** identify one function, a caller and a child; explain whether its cost lies mostly in self or descendants. Inspect the source to propose a reason, without treating a source-text match as per-line timing.

**Predict:** function `step()` is credited with 70% including children, and its child `forces()` with 60%. Is their combined contribution 130%? Explain the overlap.


<a id="worksheet"></a>
## Results worksheet

ARE job / hostname: …  Allocation: **12 cores, one node**

Module versions / executable paths: …  Results directory: …

Mesh / iteration cap / actual iterations / ranks: …

Numerical checks: …  Timing interval: …

| Observation | Measured value | Saved evidence | Interpretation / limitation |
|---|---|---|---|
| Baseline median and range | | `baseline-*.log` | |
| Sampling period / top self costs | | `gmon.PID`, `flat-profile.txt` | |
| Call counts / self and descendant costs | | `call-graph.txt` | |
| Caller–callee relationship | | `call-graph.svg` (optional) | |
| Proposed change and validation | | Source location | |


## Check your understanding

1. Which parts of gprof's evidence come from counting and which from sampling?
2. Why is `-g` insufficient to generate a classic gprof call graph?
3. Why can flat-profile cumulative seconds differ from time including children?
4. Why should call-graph percentages not be added across parent and child?
5. What assumption is used when a callee has several callers?
6. Why keep an executable without `-pg` for timing?
7. Why does this lesson disable OpenMP even though ARE has 12 cores?

<details><summary>Reference answers — open after discussing</summary>

1. Instrumentation counts observed call arcs; sampling estimates self time. Descendant time is inferred.
2. `-g` supplies debug information. `-pg` is needed during compilation and linking to arrange instrumentation and collection.
3. Cumulative seconds adds preceding flat-table rows; it does not follow the call graph.
4. Descendant costs are already included in ancestors, so costs overlap.
5. Average cost per call is assumed independent of the caller; input-dependent work can violate this.
6. Instrumentation affects the executable and execution cost. Improvements need repeated uninstrumented timings.
7. Classic gprof's counting/sampling has threading limitations; the main work uses one application thread per process. Core allocation is a resource limit, not a requirement to run twelve workers.

</details>
